# De la pregunta física a los objetos reconstruidos

Cuaderno pedagógico del proyecto de Laboratorio III. Esta primera entrega contiene solamente las secciones 0 y 1. El objetivo es comprender y defender el análisis existente antes de ampliarlo.

**Ruta futura:** coordenadas de ATLAS → $p_T,\theta,\phi,\eta$ → separaciones angulares → selección → pesos MC → histogramas → incertidumbres.

## 0. ¿Qué pregunta responde el análisis?

**Pregunta física.** ¿En qué medida las distribuciones reconstruidas $\Delta\phi_{e\mu}$ y $|\Delta\eta_{e\mu}|$ observadas en una muestra enriquecida en $t\bar t$ son descritas por la simulación de $t\bar t$ y los fondos disponibles?

Para cada intervalo $B_k$ contamos datos y sumamos pesos Monte Carlo:

$$N_k^{\rm datos}=\sum_{i\in B_k}1,\qquad N_k^{\rm MC}=\sum_{i\in B_k}w_i,\qquad R_k=\frac{N_k^{\rm datos}}{N_k^{\rm MC}}.$$

$R_k\approx1$ indica valores centrales próximos en ese intervalo; no demuestra que el modelo sea verdadero. Para comparar solamente la forma usamos $p_k=N_k/\sum_jN_j$, donde $j$ recorre todos los intervalos del mismo histograma.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

datos = np.array([95, 120, 150, 180])
mc = np.array([100, 115, 145, 185])
bins = np.arange(len(datos))

print('Datos/MC:', np.round(datos / mc, 3))
print('Forma datos:', np.round(datos / datos.sum(), 3))
print('Forma MC:   ', np.round(mc / mc.sum(), 3))

plt.step(bins, mc, where='mid', label='MC toy')
plt.plot(bins, datos, 'ko', label='Datos toy')
plt.xlabel('Intervalo angular k')
plt.ylabel('Eventos')
plt.legend(); plt.show()

**Conexión con el repositorio.** `select_events` construye la muestra; `observables` calcula los dos ángulos; `event_weights` obtiene $w_i$; `histogram` agrupa los eventos; `plots_and_summary.py` compara datos y MC.

**Comprobación 0.**
1. ¿Por qué una muestra enriquecida en $t\bar t$ no es una muestra pura?
2. ¿Qué diferencia hay entre un conteo, una forma normalizada y datos/MC?

## 1. Del proceso físico a los objetos reconstruidos

Los protones contienen quarks y gluones. Dos partones pueden producir el par:

$$pp\to t\bar t+X,\qquad t\to bW^+,\qquad \bar t\to\bar bW^-.$$

En el canal $e\mu$, los bosones $W$ producen leptones de cargas opuestas y neutrinos:

$$t\bar t\to b\bar b\,e^{\pm}\mu^{\mp}\nu\bar\nu.$$

El detector reconstruye candidatos a electrón y muón. Los quarks $b$ forman hadrones y sus productos se reconstruyen como jets; un **b-tag** indica compatibilidad con un hadrón que contiene un quark bottom, no una identificación perfecta. Los neutrinos no se detectan directamente y contribuyen al momento transversal faltante.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')
etapas = [('pp', 0.05), ('$t\\bar t$', 0.25), ('$bW^+\\,\\bar bW^-$', 0.48), ('$e^\\pm\\mu^\\mp + \\mathrm{jets} + p_T^{miss}$', 0.78)]
for texto, x in etapas:
    ax.text(x, .5, texto, ha='center', va='center', fontsize=13,
            bbox=dict(boxstyle='round', facecolor='#e8f1f8'))
for (_, x1), (_, x2) in zip(etapas[:-1], etapas[1:]):
    ax.annotate('', (x2-.07, .5), (x1+.06, .5), arrowprops=dict(arrowstyle='->'))
ax.set_title('Proceso físico → firma reconstruida')
plt.show()

In [ ]:
evento = {
    'electron': {'pt': 48., 'charge': +1},
    'muon': {'pt': 41., 'charge': -1},
    'jets': [{'pt': 72., 'b_tag': True}, {'pt': 54., 'b_tag': True}]
}
cargas_opuestas = evento['electron']['charge'] * evento['muon']['charge'] < 0
leptones_aceptados = evento['electron']['pt'] > 25 and evento['muon']['pt'] > 25
dos_btags = sum(jet['b_tag'] for jet in evento['jets']) >= 2
print('Supera el ejemplo de selección:', cargas_opuestas and leptones_aceptados and dos_btags)
print('Esto indica compatibilidad con la firma, no demuestra que sea ttbar.')

### Correspondencia con `analysis.py`

| Física | Ramas o función |
|---|---|
| Electrón y muón | `lep_type`, `lep_pt`, `lep_eta`, `lep_phi`, `lep_charge` |
| Calidad y trigger | `lep_isTightID`, `lep_isTightIso`, `lep_isTrigMatched` |
| Jets y etiquetas b | `jet_pt`, `jet_eta`, `jet_jvt`, `jet_btag_quantile` |
| Momento faltante | `met`, `met_phi` |
| Decisión por evento | `select_events` |

Los ROOT educativos ya contienen objetos reconstruidos: este proyecto no procesa las señales electrónicas originales del detector. Además, superar la selección no identifica con certeza el proceso que originó una colisión real.

**Comprobación 1.**
1. ¿Por qué un fondo puede superar la misma selección que la señal?
2. ¿Qué observa el análisis en lugar del quark $b$ libre?
3. ¿Por qué el momento faltante no es una medición directa de los neutrinos?